# Olist 챗봇용 데이터 전처리 (2판, 분석 기간 2018-08-15까지)

원본 9개 테이블을 챗봇 기능별로 바로 쓸 수 있는 **정리 테이블(마트)**로 만들어 저장하는 노트북이에요.

| 파일 | 한 행 | 쓰는 챗봇 기능 |
| --- | --- | --- |
| `orders_mart.parquet` | 주문 | 1 배송지연·쿠폰, 2 평점, 3 분석 질문, 6 블랙프라이데이 |
| `order_seller_mart.parquet` | 주문 × 판매자 | 1 배송지연·쿠폰 (판매자 진단) |
| `item_mart.parquet` | 상품 한 줄 | 4 발주, 5 매출, 6 블랙프라이데이 |
| `sales_daily.parquet` | 날짜 × 카테고리 | 5 매출, 6 블랙프라이데이 |
| `sales_weekly_seller_cat.parquet` | 주 × 판매자 × 카테고리 | 4 발주 (수요 예측) |
| `product_stats.parquet`, `seller_stats.parquet` | 상품 / 판매자 | 1 쿠폰(나쁜 리뷰 손해), 4 발주, 판매자 진단 |
| `review_text.parquet` | 리뷰 | 2 평점 + 텍스트, 3 분석 질문 |
| `events.parquet` | 이벤트 | 4 발주, 6 블랙프라이데이 |
| `event_multiples.parquet` | 이벤트 × 연도 × 카테고리 | 4 발주, 6 블랙프라이데이 (이벤트 때 몇 배 팔렸나) |
| `exclusion_log.json` | | 빠진 행·표시한 행의 개수와 이유 (발표용 데이터 품질 근거) |
| `summaries.json` | | 3 분석 질문, 5, 6에서 챗봇이 바로 꺼내 쓸 결과 표 |
| `data_dictionary.json` | | 열 설명 (챗봇 LLM이 읽고 pandas 코드를 짤 때 참고) |

**Colab 실행법**: `archive.zip`을 Colab에 올리거나 구글 드라이브 `내 드라이브/archive.zip`에 두고, **런타임 → 모두 실행**. 결과는 드라이브가 연결돼 있으면 `내 드라이브/olist_mart/`에, 아니면 Colab의 `olist_mart/` 폴더에 저장되고 zip으로도 묶여요.

**2판에서 바뀐 것**
1. 리뷰가 **물건 도착 전에** 쓰였는지 표시 (`review_before_delivery`). 이런 리뷰는 대부분 약속일까지 못 받은 고객이 쓴 거예요.
2. 위경도 중 **브라질 밖 좌표를 빼고** 우편번호 평균 위치를 계산 (판매자-고객 거리 오류 수정)
3. **판매자 과거 이력**(그 주문 이전에 배송이 끝난 주문, 이전에 달린 리뷰만)과 **주문×판매자별 무게·부피**를 `order_seller_mart`에 추가
4. **이벤트 배수 표**(`event_multiples`)를 연도·카테고리별로 한 번 계산해서 저장 (정의 통일)
5. **빠진 행·표시한 행 기록**(`exclusion_log.json`)과 **무료 배송 표시**(`free_shipping`)

**두 가지 기준 (8/15)**
- **연구 분석**(`summaries.json`, 시장 전체 요약): 8/15까지 **구매한** 주문의 결과(배송·리뷰)를 끝까지 반영해요. 8/15 뒤에 도착하거나 리뷰가 달린 주문을 빼면 늦게 오는 주문이 골라서 빠져서 결과가 좋아 보이는 착시가 생겨요.
- **챗봇의 현재 상태**(`seller_stats`, `product_stats`의 별점, 도구의 판매자 진단·리뷰 요약·브리핑): 8/15을 '오늘'로 보고 **그날까지 도착·답변된** 정보만 써요. `delivered_by_asof`, `review_visible_asof` 열로 구분해요.

**공통 규칙**
- 기간: 시기별 분석은 2017-01-01 ~ 2018-08-15 (`in_period` 열로 표시, 행은 지우지 않음). 8월 하순은 데이터 수집이 끊겨 주문이 다 들어오지 않아서 뺐어요. 주간 표는 다 채워진 주(8/12까지), 월별 증감은 다 채워진 달(7월까지)만 써요.
- 매출 = 상품가(price) 합. 운송비는 따로. 취소·판매불가(canceled, unavailable) 주문은 매출에서 제외 (`is_sale` 열)
- 배송·리뷰 분석은 배송 완료(`is_delivered`) 주문만
- 리뷰는 주문당 마지막 하나, 고객은 `customer_unique_id`로

## 0. 준비

In [1]:
import os, sys, glob, json, zipfile, unicodedata, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
IN_COLAB = 'google.colab' in sys.modules

# 설정값 (필요하면 여기만 바꾸면 됨)
PERIOD_START, PERIOD_END = '2017-01-01', '2018-08-15'   # 시기별 분석 기간 (끝 날짜 포함)
# 8월 하순부터 데이터 수집이 끊겨서 주문이 다 들어오지 않았어요 (하루 250건 → 8/29 14건 → 8/31 1건).
# 그래서 8/15까지만 쓰고, 주간 표는 다 채워진 주(일요일 8/12까지), 월별 증감은 다 채워진 달(7월까지)만 써요.
ASOF_END = pd.Timestamp(PERIOD_END) + pd.Timedelta(days=1)   # 챗봇의 '오늘' = 8/15 하루 끝. 이 시각 전에 생긴 일만 '알 수 있던 정보'
LOW_SCORE = 2                    # 이 점수 이하를 '나쁜 리뷰'로 봄
REVIEW_SALES_DROP = 0.061        # 나쁜 리뷰 뒤 4주 판매 감소율 (리뷰 전후 비교 분석 결과, 주문 20건 이상 상품 기준)
MIN_ORDERS_RELIABLE = 30         # 판매자 통계를 믿을 만한 최소 주문 수
BRAZIL_BOX = {'lat': (-34.0, 5.5), 'lng': (-74.0, -34.0)}   # 브라질 위경도 범위 (이 밖은 오류로 봄)

EXCLUSIONS = []                  # 빠지거나 표시된 행 기록 (exclusion_log.json)
def log(step, count, action, reason):
    EXCLUSIONS.append({'step': step, 'count': int(count), 'action': action, 'reason': reason})

In [2]:
# 데이터 불러오기: archive.zip → olist_data 폴더
DATA_DIR = 'olist_data'
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('드라이브 연결 건너뜀:', e)
if not os.path.exists(f'{DATA_DIR}/olist_orders_dataset.csv'):
    candidates = ['archive.zip', '/content/archive.zip', '/content/drive/MyDrive/archive.zip']
    zip_path = next((p for p in candidates if os.path.exists(p)), None)
    if zip_path is None and IN_COLAB:
        from google.colab import files
        print('archive.zip 파일을 선택해서 올려주세요')
        zip_path = list(files.upload())[0]
    zipfile.ZipFile(zip_path).extractall(DATA_DIR)

# 저장 위치: 드라이브가 연결돼 있으면 드라이브에
OUT_DIR = '/content/drive/MyDrive/olist_mart' if os.path.exists('/content/drive/MyDrive') else 'olist_mart'
os.makedirs(OUT_DIR, exist_ok=True)
print('데이터:', sorted(os.listdir(DATA_DIR)))
print('저장 위치:', OUT_DIR)

데이터: ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']
저장 위치: olist_mart


## 1. 원본 읽기 + 공통 정리

In [3]:
def read(name, **kw):
    return pd.read_csv(f'{DATA_DIR}/olist_{name}_dataset.csv', **kw)

DATE_COLS = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
orders = read('orders', parse_dates=DATE_COLS)
customers = read('customers')
items = read('order_items', parse_dates=['shipping_limit_date'])
products = read('products')
sellers = read('sellers')
payments = read('order_payments')
reviews_raw = read('order_reviews', parse_dates=['review_creation_date', 'review_answer_timestamp'])
geo = read('geolocation')
cat_tr = pd.read_csv(f'{DATA_DIR}/product_category_name_translation.csv')
for name, t in [('orders', orders), ('customers', customers), ('items', items), ('products', products), ('sellers', sellers),
                ('payments', payments), ('reviews', reviews_raw), ('geolocation', geo)]:
    print(f'{name:12s} {t.shape}')

orders       (99441, 8)
customers    (99441, 5)
items        (112650, 7)
products     (32951, 9)
sellers      (3095, 4)
payments     (103886, 5)
reviews      (99224, 7)
geolocation  (1000163, 5)


In [4]:
# 카테고리: 포르투갈어 → 영어 → 한국어(챗봇 표시용)
EXTRA_EN = {'pc_gamer': 'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos': 'portable_kitchen_food_preparers'}
CAT_KO = {
    'agro_industry_and_commerce': '농업·산업용품', 'air_conditioning': '냉난방기', 'art': '미술품', 'arts_and_craftmanship': '공예',
    'audio': '오디오', 'auto': '자동차용품', 'baby': '유아용품', 'bed_bath_table': '침구·욕실·식탁',
    'books_general_interest': '일반 도서', 'books_imported': '수입 도서', 'books_technical': '전문 도서',
    'cds_dvds_musicals': 'CD·DVD·뮤지컬', 'christmas_supplies': '크리스마스 용품', 'cine_photo': '영상·사진',
    'computers': '컴퓨터', 'computers_accessories': '컴퓨터 액세서리', 'consoles_games': '게임기·게임',
    'construction_tools_construction': '건설 공구', 'construction_tools_lights': '조명 공구', 'construction_tools_safety': '안전 공구',
    'cool_stuff': '쿨 아이템', 'costruction_tools_garden': '정원 공구', 'costruction_tools_tools': '일반 공구',
    'diapers_and_hygiene': '기저귀·위생용품', 'drinks': '음료', 'dvds_blu_ray': 'DVD·블루레이', 'electronics': '전자제품',
    'fashio_female_clothing': '여성 의류', 'fashion_bags_accessories': '가방·액세서리', 'fashion_childrens_clothes': '아동 의류',
    'fashion_male_clothing': '남성 의류', 'fashion_shoes': '신발', 'fashion_sport': '스포츠 의류',
    'fashion_underwear_beach': '속옷·수영복', 'fixed_telephony': '유선전화', 'flowers': '꽃', 'food': '식품', 'food_drink': '식음료',
    'furniture_bedroom': '침실 가구', 'furniture_decor': '가구·인테리어', 'furniture_living_room': '거실 가구',
    'furniture_mattress_and_upholstery': '매트리스·소파', 'garden_tools': '정원용품', 'health_beauty': '건강·뷰티',
    'home_appliances': '가전제품', 'home_appliances_2': '가전제품 2', 'home_comfort_2': '홈 편의용품 2', 'home_confort': '홈 편의용품',
    'home_construction': '주택 건축자재', 'housewares': '생활용품', 'industry_commerce_and_business': '산업·사무용품',
    'kitchen_dining_laundry_garden_furniture': '주방·세탁·정원 가구', 'la_cuisine': '주방용품(프리미엄)',
    'luggage_accessories': '여행가방', 'market_place': '마켓플레이스', 'music': '음악', 'musical_instruments': '악기',
    'office_furniture': '사무용 가구', 'party_supplies': '파티용품', 'perfumery': '향수', 'pet_shop': '반려동물용품',
    'security_and_services': '보안·서비스', 'signaling_and_security': '안전표지·보안', 'small_appliances': '소형가전',
    'small_appliances_home_oven_and_coffee': '오븐·커피 소형가전', 'sports_leisure': '스포츠·레저', 'stationery': '문구',
    'tablets_printing_image': '태블릿·프린터', 'telephony': '휴대폰·통신', 'toys': '장난감', 'watches_gifts': '시계·선물',
    'pc_gamer': '게이밍 PC', 'portable_kitchen_food_preparers': '휴대용 주방가전', 'unknown': '미분류',
}
products = products.merge(cat_tr, on='product_category_name', how='left')
products['category_en'] = (products.product_category_name_english
                           .fillna(products.product_category_name.map(EXTRA_EN)).fillna('unknown'))
products['category_ko'] = products.category_en.map(CAT_KO).fillna(products.category_en)
products['volume_cm3'] = products.product_length_cm * products.product_height_cm * products.product_width_cm
print('카테고리 수:', products.category_en.nunique(), '| 미분류 상품:', (products.category_en == 'unknown').sum())
log('products', (products.category_en == 'unknown').sum(), '미분류로 표시', '카테고리 정보가 없는 상품')
print('한국어 이름이 없는 카테고리:', sorted(set(products.category_en) - set(CAT_KO)))

카테고리 수: 74 | 미분류 상품: 610
한국어 이름이 없는 카테고리: []


In [5]:
# 리뷰: 주문당 마지막에 답한 리뷰 하나만
reviews = (reviews_raw.sort_values('review_answer_timestamp')
                      .drop_duplicates('order_id', keep='last'))
print('리뷰', len(reviews_raw), '→', len(reviews), '(주문당 하나)')
log('reviews', len(reviews_raw) - len(reviews), '삭제', '한 주문에 리뷰가 여러 개 → 마지막에 답한 리뷰만 남김')

# 지역: 주 → 5개 권역
REGION = {**{s: '북부' for s in ['AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO']},
          **{s: '북동부' for s in ['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE']},
          **{s: '중서부' for s in ['DF', 'GO', 'MT', 'MS']},
          **{s: '남동부' for s in ['ES', 'MG', 'RJ', 'SP']},
          **{s: '남부' for s in ['PR', 'RS', 'SC']}}
customers['customer_region'] = customers.customer_state.map(REGION)
sellers['seller_region'] = sellers.seller_state.map(REGION)

# 우편번호 앞자리별 평균 위경도 (거리 계산용). 브라질 밖 좌표는 오류라서 빼고 평균
in_box = geo.geolocation_lat.between(*BRAZIL_BOX['lat']) & geo.geolocation_lng.between(*BRAZIL_BOX['lng'])
print('브라질 밖 좌표:', int((~in_box).sum()), '개 (우편번호', geo[~in_box].geolocation_zip_code_prefix.nunique(), '곳)')
log('geolocation', (~in_box).sum(), '삭제', '브라질 위경도 범위 밖 좌표 (바다·다른 나라로 찍힌 오류)')
zip_geo = geo[in_box].groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean()
def haversine_km(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    h = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(h))

WEEKDAY_KO = ['월', '화', '수', '목', '금', '토', '일']
DELAY_BINS, DELAY_LABELS = [-1000, -1, 0, 1, 3, 5, 1000], ['일찍 옴', '당일', '1일 늦음', '2~3일 늦음', '4~5일 늦음', '6일+ 늦음']
DAYS_BINS, DAYS_LABELS = [-1, 5, 10, 15, 20, 30, 1000], ['0~5일', '5~10일', '10~15일', '15~20일', '20~30일', '30일+']

리뷰 99224 → 98673 (주문당 하나)
브라질 밖 좌표: 42 개 (우편번호 21 곳)


## 2. 이벤트 표 (`events`)
발주(4)와 블랙프라이데이(6) 분석에 쓰는 브라질 쇼핑 이벤트예요. `window_start`~`window_end`가 그 이벤트의 영향 기간(이벤트 전 일주일 등)이에요.

In [6]:
events = pd.DataFrame([
    ('black_friday',   '블랙프라이데이',        '2017-11-24', '2017-11-24', '2017-11-27'),   # 끝 = 사이버먼데이
    ('christmas',      '크리스마스',            '2017-12-25', '2017-12-11', '2017-12-24'),
    ('mothers_day',    '어머니날',              '2017-05-14', '2017-05-07', '2017-05-14'),
    ('mothers_day',    '어머니날',              '2018-05-13', '2018-05-06', '2018-05-13'),
    ('valentines_day', '연인의 날(6/12)',       '2017-06-12', '2017-06-05', '2017-06-12'),
    ('valentines_day', '연인의 날(6/12)',       '2018-06-12', '2018-06-05', '2018-06-12'),
    ('fathers_day',    '아버지날',              '2017-08-13', '2017-08-06', '2017-08-13'),
    ('fathers_day',    '아버지날',              '2018-08-12', '2018-08-05', '2018-08-12'),
], columns=['event', 'event_ko', 'date', 'window_start', 'window_end'])
for col in ['date', 'window_start', 'window_end']:
    events[col] = pd.to_datetime(events[col])

# 날짜별 이벤트 표시 (그 날짜가 어떤 이벤트 기간에 속하는지)
day_event = {}
for _, e in events.iterrows():
    for d in pd.date_range(e.window_start, e.window_end):
        day_event[d] = e.event
events

,event,event_ko,date,window_start,window_end
0,black_friday,블랙프라이데이,2017-11-24,2017-11-24,2017-11-27
1,christmas,크리스마스,2017-12-25,2017-12-11,2017-12-24
2,mothers_day,어머니날,2017-05-14,2017-05-07,2017-05-14
3,mothers_day,어머니날,2018-05-13,2018-05-06,2018-05-13
4,valentines_day,연인의 날(6/12),2017-06-12,2017-06-05,2017-06-12
5,valentines_day,연인의 날(6/12),2018-06-12,2018-06-05,2018-06-12
6,fathers_day,아버지날,2017-08-13,2017-08-06,2017-08-13
7,fathers_day,아버지날,2018-08-12,2018-08-05,2018-08-12


## 3. 주문 단위로 먼저 묶기
상품 표와 결제 표는 주문 하나에 여러 행이 있어요. 그대로 합치면 행이 늘어 매출이 중복 계산되니까, **각각 주문 단위로 묶은 뒤** 합쳐요.

In [7]:
print('주의: 상품 표와 결제 표를 바로 합치면', len(items.merge(payments, on='order_id')), '행 (상품 표는', len(items), '행)')

items_p = items.merge(products[['product_id', 'category_en', 'category_ko', 'product_weight_g', 'volume_cm3']], on='product_id', how='left')
items_p['free_shipping'] = items_p.freight_value == 0
log('order_items', items_p.free_shipping.sum(), 'free_shipping 열로 표시', '운송비 0원 (무료 배송). 운송비 기준 쿠폰은 0원이 됨')
order_items = items_p.sort_values('order_item_id').groupby('order_id').agg(
    n_items=('order_item_id', 'size'), n_products=('product_id', 'nunique'), n_sellers=('seller_id', 'nunique'),
    order_value=('price', 'sum'), freight_value=('freight_value', 'sum'),
    main_category_en=('category_en', 'first'), main_category_ko=('category_ko', 'first'))
order_pay = payments.sort_values('payment_sequential').groupby('order_id').agg(
    payment_type=('payment_type', 'first'), n_payment_types=('payment_type', 'nunique'),
    installments=('payment_installments', 'max'), payment_value=('payment_value', 'sum'))
print('order_items:', order_items.shape, '| order_pay:', order_pay.shape)

주의: 상품 표와 결제 표를 바로 합치면 117601 행 (상품 표는 112650 행)


order_items: (98666, 7) | order_pay: (99440, 4)


## 4. `orders_mart` (한 행 = 주문)

In [8]:
om = (orders.merge(customers[['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
                              'customer_state', 'customer_region']], on='customer_id', how='left')
            .merge(order_items, on='order_id', how='left')
            .merge(order_pay, on='order_id', how='left')
            .merge(reviews[['order_id', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message',
                            'review_creation_date', 'review_answer_timestamp']], on='order_id', how='left'))
day = pd.Timedelta(days=1)
ts = om.order_purchase_timestamp
om = om.assign(
    purchase_date=ts.dt.normalize(),
    year_month=ts.dt.strftime('%Y-%m'),
    week_start=(ts.dt.normalize() - pd.to_timedelta(ts.dt.dayofweek, unit='D')),     # 그 주 월요일
    weekday=ts.dt.dayofweek, weekday_ko=ts.dt.dayofweek.map(dict(enumerate(WEEKDAY_KO))), hour=ts.dt.hour,
    in_period=(ts >= PERIOD_START) & (ts < pd.Timestamp(PERIOD_END) + day),
    is_sale=~om.order_status.isin(['canceled', 'unavailable']) & om.order_value.notna(),
    is_delivered=(om.order_status == 'delivered') & om.order_delivered_customer_date.notna(),
    approve_hours=(om.order_approved_at - ts) / pd.Timedelta(hours=1),
    promised_days=(om.order_estimated_delivery_date - ts) / day,
)
dlv = om.is_delivered
om['days'] = ((om.order_delivered_customer_date - ts) / day).where(dlv)
om['delay_days'] = (om.order_delivered_customer_date.dt.normalize() - om.order_estimated_delivery_date.dt.normalize()).dt.days.where(dlv)
om['late'] = (om.delay_days > 0).where(dlv).astype('boolean')
om['delay_bin'] = pd.cut(om.delay_days, DELAY_BINS, labels=DELAY_LABELS)
om['days_bin'] = pd.cut(om.days, DAYS_BINS, labels=DAYS_LABELS)
om['long_delivery_flag'] = om.days > 60                      # 이상치 표시 (지우지 않음)
om['has_review'] = om.review_score.notna()
om['low_review'] = (om.review_score <= LOW_SCORE).where(om.has_review).astype('boolean')
om['has_text'] = om.review_comment_message.notna()
om['answer_hours'] = (om.review_answer_timestamp - om.review_creation_date) / pd.Timedelta(hours=1)
om['event'] = om.purchase_date.map(day_event)
# 리뷰 요청이 약속일 무렵에 나가서, 그때까지 못 받은 고객은 도착 전에 리뷰를 씀
om['review_before_delivery'] = (om.review_creation_date < om.order_delivered_customer_date.dt.normalize()).where(dlv & om.has_review).astype('boolean')
log('orders', om.long_delivery_flag.sum(), 'long_delivery_flag 열로 표시', '배송 60일 초과 (지우지 않음)')
log('orders', (~om.in_period).sum(), 'in_period=False로 표시', f'{PERIOD_START} ~ {PERIOD_END} 밖 (시기별 분석에서 제외)')
log('orders', ((om.order_purchase_timestamp > pd.Timestamp(PERIOD_END) + day) & (om.order_purchase_timestamp < '2018-09-01')).sum(),
    'in_period=False로 표시', '2018-08-16 ~ 08-31: 데이터 수집이 끊겨 주문이 다 들어오지 않은 구간')
log('orders', (~om.is_delivered).sum(), 'is_delivered=False로 표시', '배송 완료가 아님 (취소, 배송 중 등). 배송·리뷰 분석에서 제외')
log('orders', om.review_before_delivery.fillna(False).sum(), 'review_before_delivery 열로 표시', '물건 도착 전에 쓰인 리뷰')
om['delivered_by_asof'] = om.order_delivered_customer_date < ASOF_END          # 8/15까지 고객에게 도착했나
om['review_visible_asof'] = om.review_answer_timestamp < ASOF_END                  # 8/15까지 리뷰 답변이 달렸나 (판매자가 볼 수 있었나)
log('orders', (om.in_period & om.is_delivered & ~om.delivered_by_asof).sum(), 'delivered_by_asof=False로 표시',
    '기간 안에 샀지만 8/16 이후 도착 (연구 분석에는 포함, 챗봇 현재 상태에서는 제외)')
log('orders', (om.in_period & om.has_review & ~om.review_visible_asof).sum(), 'review_visible_asof=False로 표시',
    '기간 안에 샀지만 리뷰 답변이 8/16 이후 (연구 분석에는 포함, 챗봇 현재 상태에서는 제외)')
om = om.drop(columns=['review_comment_message', 'review_comment_title'])   # 글은 review_text 표에 따로

assert om.order_id.is_unique and len(om) == len(orders), '주문 수가 바뀌었어요'
orders_mart = om
print(orders_mart.shape)
orders_mart[['order_id', 'year_month', 'customer_region', 'n_items', 'order_value', 'days', 'promised_days',
             'delay_days', 'delay_bin', 'review_score', 'event']].head()

(99441, 53)


,order_id,year_month,customer_region,n_items,order_value,days,promised_days,delay_days,delay_bin,review_score,event
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10,남동부,1.0,29.99,8.436574,15.544063,-8.0,일찍 옴,4.0,NaN
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07,북동부,1.0,118.70,13.782037,19.137766,-6.0,일찍 옴,4.0,NaN
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08,중서부,1.0,159.90,9.394213,26.639711,-18.0,일찍 옴,5.0,fathers_day
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11,북동부,1.0,45.00,13.208750,26.188819,-13.0,일찍 옴,5.0,NaN
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02,남동부,1.0,19.90,2.873877,12.112049,-10.0,일찍 옴,5.0,NaN


## 5. `order_seller_mart` (한 행 = 주문 × 판매자)
판매자 몫(승인 → 택배사 인계)과 택배 몫(인계 → 도착)을 나누고, 발송 마감을 지켰는지 표시해요.

In [9]:
osm = items_p.groupby(['order_id', 'seller_id']).agg(
    n_items=('order_item_id', 'size'), price=('price', 'sum'), freight_value=('freight_value', 'sum'),
    weight_g=('product_weight_g', 'sum'), volume_cm3=('volume_cm3', 'sum'),
    category_en=('category_en', 'first'), category_ko=('category_ko', 'first'),
    shipping_limit=('shipping_limit_date', 'max')).reset_index()
osm['free_shipping'] = osm.freight_value == 0
osm = (osm.merge(orders_mart[['order_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at',
                              'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date',
                              'customer_unique_id', 'customer_zip_code_prefix', 'customer_state', 'customer_region',
                              'year_month', 'in_period', 'is_sale', 'is_delivered', 'promised_days', 'days',
                              'delay_days', 'late', 'delay_bin', 'review_score', 'low_review', 'event',
                              'delivered_by_asof', 'review_visible_asof']], on='order_id', how='left')
          .merge(sellers[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'seller_region']], on='seller_id', how='left'))
osm['handling_days'] = (osm.order_delivered_carrier_date - osm.order_approved_at) / day     # 판매자 몫
osm['transit_days'] = (osm.order_delivered_customer_date - osm.order_delivered_carrier_date) / day   # 택배 몫
osm['ship_late'] = (osm.order_delivered_carrier_date > osm.shipping_limit).where(osm.order_delivered_carrier_date.notna()).astype('boolean')
osm['valid_times'] = osm.is_delivered & (osm.handling_days >= 0) & (osm.transit_days >= 0)
dl = osm[osm.is_delivered]
log('order_seller', dl.order_approved_at.isna().sum(), 'valid_times=False', '배송 완료인데 주문 승인 시각이 없음')
log('order_seller', dl.order_delivered_carrier_date.isna().sum(), 'valid_times=False', '배송 완료인데 택배사 인계 시각이 없음')
log('order_seller', (dl.handling_days < 0).sum(), 'valid_times=False', '택배사 인계가 주문 승인보다 앞섬 (기록 오류)')
log('order_seller', (dl.transit_days < 0).sum(), 'valid_times=False', '고객 도착이 택배사 인계보다 앞섬 (기록 오류)')
osm['same_state'] = osm.customer_state == osm.seller_state
c = zip_geo.reindex(osm.customer_zip_code_prefix).to_numpy()
s = zip_geo.reindex(osm.seller_zip_code_prefix).to_numpy()
osm['dist_km'] = haversine_km(s[:, 0], s[:, 1], c[:, 0], c[:, 1])

assert not osm.duplicated(['order_id', 'seller_id']).any()
print('거리 최댓값(km):', round(osm.dist_km.max()), '| 거리 계산 안 된 비율:', round(osm.dist_km.isna().mean(), 4))
order_seller_mart = osm
print(order_seller_mart.shape, '| 시간 값이 정상인 배송 완료 행:', int(osm.valid_times.sum()))
order_seller_mart[['order_id', 'seller_id', 'category_ko', 'handling_days', 'transit_days', 'ship_late', 'late', 'dist_km']].head()

거리 최댓값(km): 3580 | 거리 계산 안 된 비율: 0.005
(100010, 45) | 시간 값이 정상인 배송 완료 행: 96365


,order_id,seller_id,category_ko,handling_days,transit_days,ship_late,late,dist_km
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,쿨 아이템,6.367141,1.214954,True,False,301.504681
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,반려동물용품,8.145683,8.062083,True,False,585.563937
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,가구·인테리어,1.908542,6.029491,False,False,312.343511
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,향수,2.137292,4.003229,False,False,293.168420
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,정원용품,11.816620,13.289144,True,False,646.163463


### 판매자 과거 이력 (미래 정보 누수 없이)
"이 판매자는 전에 얼마나 늦었나"를 예측 입력으로 쓰려면, **그 주문의 구매 시각 이전에 이미 배송이 끝난 주문**, **이전에 달린 리뷰**만 세야 해요. 그냥 누적하면 아직 결과를 알 수 없는 주문까지 섞여요. `merge_asof`로 판매자별 "그 시각까지의 누적값"을 붙여요.

In [10]:
def past_stats(events, time_col, value_col, prefix):
    e = events[['seller_id', time_col, value_col]].dropna().sort_values(time_col).copy()
    e[f'{prefix}_n'] = e.groupby('seller_id').cumcount() + 1
    e[f'{prefix}_sum'] = e.groupby('seller_id')[value_col].cumsum()
    left = order_seller_mart[['seller_id', 'order_purchase_timestamp']].reset_index().sort_values('order_purchase_timestamp')
    m = pd.merge_asof(left, e[['seller_id', time_col, f'{prefix}_n', f'{prefix}_sum']].sort_values(time_col),
                      left_on='order_purchase_timestamp', right_on=time_col, by='seller_id',
                      direction='backward', allow_exact_matches=False)     # 구매 시각보다 '앞선' 사건만
    return m.set_index('index').sort_index()

ev_deliv = order_seller_mart[order_seller_mart.valid_times].assign(late_num=lambda x: x.late.astype(float))
ev_rev = order_seller_mart.merge(orders_mart[['order_id', 'review_answer_timestamp']], on='order_id', how='left')
pl = past_stats(ev_deliv, 'order_delivered_customer_date', 'late_num', 'pd')
pr = past_stats(ev_rev, 'review_answer_timestamp', 'review_score', 'pr')
order_seller_mart['seller_past_orders'] = pl.pd_n.fillna(0).astype(int)                 # 이전에 배송이 끝난 주문 수
order_seller_mart['seller_past_late_rate'] = pl.pd_sum / pl.pd_n                        # 그중 늦은 비율
order_seller_mart['seller_past_score'] = pr.pr_sum / pr.pr_n                            # 이전에 달린 리뷰 평균
order_seller_mart[['seller_past_orders', 'seller_past_late_rate', 'seller_past_score']].describe().round(3)

,seller_past_orders,seller_past_late_rate,seller_past_score
count,100010.000,94297.000,93750.000
mean,168.369,0.045,4.143
std,282.644,0.066,0.444
min,0.000,0.000,1.000
25%,11.000,0.000,3.973
50%,50.000,0.031,4.162
75%,179.000,0.062,4.368
max,1747.000,1.000,5.000


## 6. `item_mart` (한 행 = 상품 한 줄)

In [11]:
item_mart = (items_p.merge(products[['product_id', 'product_photos_qty']], on='product_id', how='left')   # 무게·부피는 items_p에 이미 있음
                    .merge(orders_mart[['order_id', 'order_status', 'order_purchase_timestamp', 'purchase_date', 'week_start',
                                        'year_month', 'in_period', 'is_sale', 'customer_state', 'customer_region', 'event']],
                           on='order_id', how='left')
                    .merge(sellers[['seller_id', 'seller_state', 'seller_region']], on='seller_id', how='left'))
assert len(item_mart) == len(items)
print(item_mart.shape)
item_mart.head(3)

(112650, 25)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,category_en,category_ko,product_weight_g,volume_cm3,free_shipping,product_photos_qty,order_status,order_purchase_timestamp,purchase_date,week_start,year_month,in_period,is_sale,customer_state,customer_region,event,seller_state,seller_region
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,cool_stuff,쿨 아이템,650.0,3528.0,False,4.0,delivered,2017-09-13 08:59:02,2017-09-13,2017-09-11,2017-09,True,True,RJ,남동부,NaN,SP,남동부
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,pet_shop,반려동물용품,30000.0,60000.0,False,2.0,delivered,2017-04-26 10:53:06,2017-04-26,2017-04-24,2017-04,True,True,SP,남동부,NaN,SP,남동부
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,furniture_decor,가구·인테리어,3050.0,14157.0,False,2.0,delivered,2018-01-14 14:33:31,2018-01-14,2018-01-08,2018-01,True,True,MG,남동부,NaN,MG,남동부


## 7. 판매 집계 표 (매출 5, 블랙프라이데이 6, 발주 4)
- `sales_daily`: 날짜 × 카테고리. **판매가 없는 날은 0으로 채워요.** 안 채우면 평균이 부풀려지고 이동평균이 틀어져요.
- `sales_weekly_seller_cat`: 주 × 판매자 × 카테고리. 판매자의 첫 판매 주부터 기간 끝까지 0으로 채워서 발주용 수요 예측에 써요.

In [12]:
sales = item_mart[item_mart.is_sale & item_mart.in_period]

daily = (sales.groupby(['purchase_date', 'category_en'])
              .agg(orders=('order_id', 'nunique'), units=('order_item_id', 'size'), revenue=('price', 'sum'), freight=('freight_value', 'sum')))
all_days = pd.date_range(PERIOD_START, PERIOD_END, freq='D')
all_cats = sorted(sales.category_en.unique())
grid = pd.MultiIndex.from_product([all_days, all_cats], names=['purchase_date', 'category_en'])
sales_daily = daily.reindex(grid, fill_value=0).reset_index()
sales_daily['category_ko'] = sales_daily.category_en.map(CAT_KO).fillna(sales_daily.category_en)
sales_daily['weekday'] = sales_daily.purchase_date.dt.dayofweek
sales_daily['event'] = sales_daily.purchase_date.map(day_event)
print('sales_daily:', sales_daily.shape)

sales_daily: (43808, 9)


In [13]:
weekly = (sales.groupby(['seller_id', 'category_en', 'week_start'])
               .agg(orders=('order_id', 'nunique'), units=('order_item_id', 'size'), revenue=('price', 'sum')).reset_index())
first_week = weekly.groupby(['seller_id', 'category_en']).week_start.min().rename('first_week').reset_index()
all_weeks = pd.DataFrame({'week_start': pd.date_range(pd.Timestamp(PERIOD_START) - pd.Timedelta(days=pd.Timestamp(PERIOD_START).dayofweek),
                                                       PERIOD_END, freq='W-MON')})
LAST_FULL_WEEK = all_weeks.week_start[all_weeks.week_start + pd.Timedelta(days=6) <= pd.Timestamp(PERIOD_END)].max()
all_weeks = all_weeks[all_weeks.week_start <= LAST_FULL_WEEK]          # 다 채워진 주만 (월~일)
weekly = weekly[weekly.week_start <= LAST_FULL_WEEK]
print('주간 표 마지막 주:', LAST_FULL_WEEK.date(), '~', (LAST_FULL_WEEK + pd.Timedelta(days=6)).date())
pairs = first_week.merge(all_weeks, how='cross')
pairs = pairs[pairs.week_start >= pairs.first_week].drop(columns='first_week')
sales_weekly_seller_cat = pairs.merge(weekly, on=['seller_id', 'category_en', 'week_start'], how='left').fillna({'orders': 0, 'units': 0, 'revenue': 0})
sales_weekly_seller_cat['category_ko'] = sales_weekly_seller_cat.category_en.map(CAT_KO).fillna(sales_weekly_seller_cat.category_en)
week_event = {w: next((day_event[d] for d in pd.date_range(w, w + pd.Timedelta(days=6)) if d in day_event), None) for w in all_weeks.week_start}
sales_weekly_seller_cat['event'] = sales_weekly_seller_cat.week_start.map(week_event)
print('sales_weekly_seller_cat:', sales_weekly_seller_cat.shape, '| (판매자, 카테고리) 쌍:', len(first_week))

주간 표 마지막 주: 2018-08-06 ~ 2018-08-12


sales_weekly_seller_cat: (251815, 8) | (판매자, 카테고리) 쌍: 6370


## 8. `product_stats`, `seller_stats`
- `bad_review_cost`: 이 상품이 1~2점 리뷰를 하나 받으면 다음 한 달에 잃을 것으로 보는 매출 (월 판매량 × 가격 × 6.1%). 주문 20건 이상 상품에서 나온 값이라 `reliable` 열로 구분해요.
- `seller_stats`: 같은 대표 카테고리 + 주문 30건 이상 판매자를 동종 그룹으로 보고 중앙값과 순위를 붙였어요.

In [14]:
period_end = pd.Timestamp(PERIOD_END)
prev_scores = orders_mart.set_index('order_id').review_score.where(orders_mart.set_index('order_id').review_visible_asof)   # 8/15까지 달린 리뷰만
ps = sales.assign(review_score=sales.order_id.map(prev_scores)).groupby('product_id').agg(
    category_en=('category_en', 'first'), category_ko=('category_ko', 'first'), n_sellers=('seller_id', 'nunique'),
    n_orders=('order_id', 'nunique'), units=('order_item_id', 'size'), revenue=('price', 'sum'), median_price=('price', 'median'),
    first_sale=('purchase_date', 'min'), last_sale=('purchase_date', 'max'),
    avg_score=('review_score', 'mean'), low_rate=('review_score', lambda x: (x <= LOW_SCORE).mean()))
weeks_listed = ((period_end - ps.first_sale) / pd.Timedelta(weeks=1)).clip(lower=1)
ps['monthly_units'] = ps.units / weeks_listed * 4.345
ps['bad_review_cost'] = ps.monthly_units * ps.median_price * REVIEW_SALES_DROP
ps['reliable'] = ps.n_orders >= 20
product_stats = ps.reset_index()
print('product_stats:', product_stats.shape, '| 믿을 만한 상품(20건 이상):', int(ps.reliable.sum()))
product_stats[product_stats.reliable].sort_values('bad_review_cost', ascending=False).head()   # 믿을 만한 상품만 보기

product_stats: (31889, 15) | 믿을 만한 상품(20건 이상): 599


,product_id,category_en,category_ko,n_sellers,n_orders,units,revenue,median_price,first_sale,last_sale,avg_score,low_rate,monthly_units,bad_review_cost,reliable
28844,e7cc48a9daff5436f63d3aad9426f28b,telephony,휴대폰·통신,1,60,89,15420.00,170.0,2018-07-22,2018-08-13,3.517241,0.101124,112.788958,1169.621498,True
7797,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,컴퓨터 액세서리,1,255,274,41082.60,149.9,2018-01-10,2018-08-09,4.200743,0.113139,38.404194,351.164105,True
23309,bb50f2e236e5eea0100680137654686c,health_beauty,건강·뷰티,1,176,184,60255.00,325.0,2017-08-18,2018-08-15,4.160714,0.103261,15.459558,306.485738,True
14445,73326828aa5efe1ba096223de496f596,food,식품,1,21,21,1730.83,82.0,2018-08-04,2018-08-15,5.000000,0.000000,58.065000,290.441130,True
4847,25c38557cf793876c5abdd5931f922db,baby,유아용품,2,38,38,38907.32,999.0,2017-12-11,2018-04-30,2.675676,0.526316,4.679231,285.147644,True


In [15]:
# 챗봇의 '현재 상태'라서 8/15까지 도착한 주문, 8/15까지 달린 리뷰만 씀
v = order_seller_mart[order_seller_mart.valid_times & order_seller_mart.in_period & order_seller_mart.delivered_by_asof].copy()
v['review_score'] = v.review_score.where(v.review_visible_asof)
v['low_review'] = v.low_review.where(v.review_visible_asof)
main_cat = items_p.groupby(['seller_id', 'category_en']).size().reset_index(name='n').sort_values('n', ascending=False).drop_duplicates('seller_id')
ss = v.groupby('seller_id').agg(
    orders=('order_id', 'nunique'), revenue=('price', 'sum'), avg_score=('review_score', 'mean'),
    low_rate=('low_review', 'mean'), late_rate=('late', 'mean'), handling_med=('handling_days', 'median'),
    transit_med=('transit_days', 'median'), ship_late_rate=('ship_late', 'mean'), multi_item_share=('n_items', lambda x: (x > 1).mean()),
    first_sale=('order_purchase_timestamp', 'min'), last_sale=('order_purchase_timestamp', 'max'))
ss = ss.join(sellers.set_index('seller_id')[['seller_state', 'seller_region', 'seller_city']]).join(main_cat.set_index('seller_id').category_en.rename('main_category_en'))
ss['main_category_ko'] = ss.main_category_en.map(CAT_KO).fillna(ss.main_category_en)
ss['reliable'] = ss.orders >= MIN_ORDERS_RELIABLE

# 동종 비교: 같은 대표 카테고리 + 믿을 만한 판매자
peer = ss[ss.reliable].groupby('main_category_en')
for m in ['avg_score', 'late_rate', 'handling_med', 'ship_late_rate', 'low_rate']:
    ss[f'peer_{m}'] = ss.main_category_en.map(peer[m].median())
ss['peer_n'] = ss.main_category_en.map(peer.size()).fillna(0).astype(int)
ss['score_rank_in_peer'] = ss[ss.reliable].groupby('main_category_en').avg_score.rank(ascending=False, method='min')
seller_stats = ss.reset_index()
print('seller_stats:', seller_stats.shape, '| 믿을 만한 판매자:', int(ss.reliable.sum()))
seller_stats.sort_values('orders', ascending=False).head(3)

seller_stats: (2828, 25) | 믿을 만한 판매자: 592


,seller_id,orders,revenue,avg_score,low_rate,late_rate,handling_med,transit_med,ship_late_rate,multi_item_share,first_sale,last_sale,seller_state,seller_region,seller_city,main_category_en,main_category_ko,reliable,peer_avg_score,peer_late_rate,peer_handling_med,peer_ship_late_rate,peer_low_rate,peer_n,score_rank_in_peer
813,4a3ca9315b744ce9f8e9374361493884,1724,192854.02,3.846969,0.18658,0.098028,1.886186,9.195330,0.088747,0.084687,2017-01-08 09:35:14,2018-08-08 16:23:28,SP,남동부,ibitinga,bed_bath_table,침구·욕실·식탁,True,3.997033,0.066054,1.925807,0.050431,0.173434,44,30.0
1125,6560211a19b47992c3666cc44a7e94c0,1659,109088.83,3.989538,0.156923,0.050633,0.934271,6.445544,0.006028,0.082580,2017-02-17 07:39:19,2018-08-13 15:05:14,SP,남동부,sao paulo,watches_gifts,시계·선물,True,4.250000,0.054054,1.864144,0.039474,0.103448,23,20.0
2274,cc419e0650a3c5ba77189a1882b7556a,1627,99522.38,4.148515,0.12995,0.053473,2.005671,6.614896,0.006761,0.034419,2017-01-31 17:15:33,2018-08-09 08:04:36,SP,남동부,santo andre,health_beauty,건강·뷰티,True,4.210526,0.056075,1.712975,0.05102,0.119048,61,40.0


## 9. `review_text` (한 행 = 리뷰)
- 글이 없는 리뷰는 빈칸(결측) 그대로 둬요. ''로 채우면 "글 있는 비율" 계산이 틀어져요.
- 리뷰 속 가게 이름은 왕좌의 게임 가문 이름(lannister, stark, targaryen, baratheon)으로 익명화돼 있어서 `[loja]`(가게)로 바꿔요.
- 불만 유형 6개는 포르투갈어 키워드 규칙이에요. 나중에 이걸 라벨로 딥러닝 분류기(BERTimbau 등)를 학습시킬 수 있어요.

In [16]:
def strip_accents(s):
    return ''.join(ch for ch in unicodedata.normalize('NFKD', s) if not unicodedata.combining(ch))

rt = reviews[['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message',
              'review_creation_date', 'review_answer_timestamp']].copy()
rt['has_text'] = rt.review_comment_message.notna()
rt['msg_clean'] = (rt.review_comment_message.str.lower()
                     .str.replace(r'\b(?:lannister|stark|targaryen|baratheon)\w*', '[loja]', regex=True)
                     .str.replace(r'\s+', ' ', regex=True).str.strip())
rt['msg_noaccent'] = rt.msg_clean.map(strip_accents, na_action='ignore')
rt['title_clean'] = rt.review_comment_title.str.lower().str.strip()

COMPLAINTS = {   # 악센트를 지운 문장(msg_noaccent)에 적용
    'c_not_received': r'nao\s+(?:recebi|chegou|foi entregue)|ainda nao|nunca chegou',
    'c_delay': r'atras|prazo|demor',
    'c_wrong_item': r'errad|diferente|veio outro|enviaram outro|mandaram outro',
    'c_defect': r'quebrad|defeit|danific|nao funciona|parou de funcionar',
    'c_missing_part': r'faltando|falta|apenas um|so (?:um|uma|1)\b|somente um|incomplet',
    'c_quality_fake': r'qualidade|falsific|replica|nao e original|pessim',
}
for col, pat in COMPLAINTS.items():
    rt[col] = rt.msg_noaccent.str.contains(pat, regex=True).where(rt.has_text).astype('boolean')
rt['said_days'] = rt.msg_noaccent.str.extract(r'(\d{1,3})\s*dias')[0].astype(float)
rt['answer_hours'] = (rt.review_answer_timestamp - rt.review_creation_date) / pd.Timedelta(hours=1)
rt = rt.merge(orders_mart[['order_id', 'is_delivered', 'in_period', 'late', 'delay_days', 'delay_bin', 'days', 'review_before_delivery',
                           'customer_state', 'customer_region', 'main_category_en', 'main_category_ko']], on='order_id', how='left')
sellers_of_order = items.groupby('order_id').seller_id.first()
rt['seller_id'] = rt.order_id.map(sellers_of_order)
rt['review_visible_asof'] = rt.review_answer_timestamp < ASOF_END
review_text = rt
print(review_text.shape, '| 글 있는 리뷰 비율:', round(rt.has_text.mean(), 3))
print('익명화 이름이 남아 있는 글:', int(rt.msg_clean.str.contains(r'lannister|stark|targaryen|baratheon', na=False).sum()))
review_text[['review_score', 'msg_clean', 'c_not_received', 'c_delay', 'said_days']].dropna(subset=['msg_clean']).head()

(98673, 32) | 글 있는 리뷰 비율: 0.413
익명화 이름이 남아 있는 글: 0


,review_score,msg_clean,c_not_received,c_delay,said_days
0,1,nao recebi o produto e nem resposta da empresa,True,False,NaN
1,1,"produto não chegou,e já passou o prazo de entrega",True,True,NaN
3,5,o pedido foi entregue antes do prazo pr0metido,False,True,NaN
4,4,só chegou uma parte do pedido ate agora..,False,False,NaN
5,5,tudo dentro do combinado,False,False,NaN


## 9-2. `event_multiples` (이벤트 × 연도 × 카테고리)
이벤트 기간에 하루 판매 수량이 평소의 몇 배였는지예요. **평소** = 이벤트 시작 전 28일 중 다른 이벤트 날짜를 뺀 날들의 하루 평균. 평소 판매가 28일 합계 20개 미만인 카테고리는 숫자가 흔들려서 비워 둬요.

발주 예측에서 2018년 블랙프라이데이를 준비할 때는 **2017년 블랙프라이데이 배수**처럼 지난해 값을 쓰면 미래 정보가 섞이지 않아요.

In [17]:
piv = sales_daily.pivot_table(index='purchase_date', columns='category_en', values='units', aggfunc='sum').fillna(0)
piv['__all__'] = piv.sum(axis=1)
rows = []
for e in events.itertuples():
    w = piv.loc[e.window_start:e.window_end]
    base_idx = [d for d in pd.date_range(e.window_start - pd.Timedelta(days=28), e.window_start - pd.Timedelta(days=1))
                if d not in day_event and d in piv.index]
    if w.empty or not base_idx: continue
    b = piv.loc[base_idx]
    for cat in piv.columns:
        ok = b[cat].sum() >= 20 and b[cat].mean() > 0
        rows.append({'event': e.event, 'event_ko': e.event_ko, 'year': e.date.year, 'date': e.date, 'category_en': cat,
                     'window_daily_units': w[cat].mean(), 'base_daily_units': b[cat].mean(),
                     'multiple': (w[cat].mean() / b[cat].mean()) if ok else np.nan})
event_multiples = pd.DataFrame(rows)
event_multiples['category_ko'] = event_multiples.category_en.map(CAT_KO).fillna(event_multiples.category_en).replace({'__all__': '전체'})
print('event_multiples:', event_multiples.shape)
event_multiples[event_multiples.category_en == '__all__'][['event_ko', 'year', 'window_daily_units', 'base_daily_units', 'multiple']].round(2)

event_multiples: (600, 9)


,event_ko,year,window_daily_units,base_daily_units,multiple
74,블랙프라이데이,2017,718.75,194.07,3.70
149,크리스마스,2017,189.93,264.54,0.72
224,어머니날,2017,125.50,96.07,1.31
299,어머니날,2018,315.75,272.64,1.16
374,연인의 날(6/12),2017,135.75,134.76,1.01
449,연인의 날(6/12),2018,255.25,214.36,1.19
524,아버지날,2017,146.88,152.75,0.96
599,아버지날,2018,319.00,248.46,1.28


## 10. `summaries.json`: 챗봇이 바로 꺼내 쓸 결과 표
분석 질문(3), 매출(5), 블랙프라이데이(6), 쿠폰 확률표(1)를 미리 계산해 둬요. 챗봇은 질문을 받으면 이 표를 꺼내 설명만 하면 돼요.

In [18]:
D = orders_mart[orders_mart.is_delivered & orders_mart.in_period]
S = orders_mart[orders_mart.is_sale & orders_mart.in_period]
R = review_text[review_text.is_delivered & review_text.in_period]
summ = {}

# (1·3) 지연 구간별 별점 = 쿠폰 계산의 '1~2점 확률' 표
summ['delay_vs_score'] = D.groupby('delay_bin', observed=False).agg(
    orders=('order_id', 'size'), mean_score=('review_score', 'mean'), low_rate=('low_review', 'mean')).round(3).reset_index().to_dict('records')
# (3) 같은 배송일, 약속 지킴 vs 어김
sd = D.pivot_table(index='days_bin', columns='late', values='review_score', aggfunc='mean', observed=False).round(2)
summ['same_days_kept_vs_broken'] = [{'days_bin': str(i), 'kept': row.get(False), 'broken': row.get(True)} for i, row in sd.iterrows()]
# (3) 주별·권역별 불만
st = D.groupby('customer_state').agg(orders=('order_id', 'size'), low_rate=('low_review', 'mean'), late_rate=('late', 'mean'),
                                     mean_score=('review_score', 'mean')).round(3).sort_values('low_rate', ascending=False)
summ['state_complaints'] = st.reset_index().to_dict('records')
summ['region_summary'] = D.groupby('customer_region').agg(median_days=('days', 'median'), late_rate=('late', 'mean'),
                                                          mean_score=('review_score', 'mean')).round(3).reset_index().to_dict('records')
# (3) 리뷰 답변 속도
q = D.dropna(subset=['answer_hours']).copy()
q['speed'] = pd.qcut(q.answer_hours, 4, labels=['가장 빠름 25%', '빠름', '느림', '가장 느림 25%'])
summ['answer_speed_vs_score'] = q.groupby('speed', observed=False).agg(median_hours=('answer_hours', 'median'), mean_score=('review_score', 'mean'),
                                                                      low_rate=('low_review', 'mean')).round(3).reset_index().to_dict('records')
# (3) 도착 전에 쓰인 리뷰: 약속일까지 못 받은 고객
rb = D[D.has_review].groupby('review_before_delivery').agg(orders=('order_id', 'size'), late_rate=('late', 'mean'),
                                                          low_rate=('low_review', 'mean'), mean_score=('review_score', 'mean')).round(3)
summ['review_before_delivery'] = {'share_of_reviews': round(float(D[D.has_review].review_before_delivery.mean()), 4),
                                  'table': [{'before_delivery': bool(k), **v} for k, v in rb.to_dict('index').items()]}
# (2·3) 리뷰에 "N일"을 쓴 고객
sdx = R[R.said_days.between(1, 120)]
summ['said_days'] = {'n': int(len(sdx)), 'median_said': float(sdx.said_days.median()), 'median_actual': float(sdx.days.median()),
                     'mean_score': round(float(sdx.review_score.mean()), 2), 'overall_mean_score': round(float(R.review_score.mean()), 2)}
# (2) 불만 유형: 약속 어긴 주문 vs 지킨 주문 (1~2점, 글 있는 리뷰)
lowR = R[(R.review_score <= LOW_SCORE) & R.has_text]
ccols = list(COMPLAINTS)
summ['complaints_late_vs_ontime'] = {'late': lowR[lowR.late == True][ccols].mean().round(3).to_dict(),
                                     'on_time': lowR[lowR.late == False][ccols].mean().round(3).to_dict()}
# (5) 월별 매출과 전월 대비 증감
last_full_month = (pd.Timestamp(PERIOD_END) + pd.Timedelta(days=1)).to_period('M') - 1 if (pd.Timestamp(PERIOD_END) + pd.Timedelta(days=1)).day != 1 else pd.Timestamp(PERIOD_END).to_period('M')
ms = S[S.year_month <= str(last_full_month)].groupby('year_month').agg(orders=('order_id', 'size'), revenue=('order_value', 'sum'))   # 다 채워진 달만
ms['revenue_mom'] = ms.revenue.pct_change().round(3)
summ['monthly_sales'] = ms.round(2).reset_index().to_dict('records')
# (5) 카테고리 순위와 비중 (파이용: 상위 10 + 기타)
cat_rev = sales.groupby('category_ko').price.sum().sort_values(ascending=False)
summ['category_rank'] = [{'category_ko': k, 'revenue': round(float(v), 2), 'share': round(float(v / cat_rev.sum()), 4)} for k, v in cat_rev.head(15).items()]
pie = pd.concat([cat_rev.head(10), pd.Series({'기타': cat_rev.iloc[10:].sum()})])
summ['category_pie_top10'] = [{'category_ko': k, 'share': round(float(v / pie.sum()), 4)} for k, v in pie.items()]
summ['category_units_rank'] = sales.groupby('category_ko').size().sort_values(ascending=False).head(15).to_dict()
print(json.dumps(summ['delay_vs_score'], ensure_ascii=False, default=str)[:400])

[{"delay_bin": "일찍 옴", "orders": 86382, "mean_score": 4.291, "low_rate": 0.093}, {"delay_bin": "당일", "orders": 1202, "mean_score": 4.011, "low_rate": 0.128}, {"delay_bin": "1일 늦음", "orders": 808, "mean_score": 3.716, "low_rate": 0.2}, {"delay_bin": "2~3일 늦음", "orders": 1031, "mean_score": 2.928, "low_rate": 0.425}, {"delay_bin": "4~5일 늦음", "orders": 894, "mean_score": 2.344, "low_rate": 0.604}, {"


In [19]:
# (6) 블랙프라이데이: 당일·BF 주간 vs 직전 4주 같은 요일
tot = sales_daily.groupby('purchase_date')[['orders', 'units', 'revenue']].sum()
bf = pd.Timestamp('2017-11-24')
base_days = [bf - pd.Timedelta(weeks=k) for k in range(1, 5)]                 # 직전 4주 금요일
bf_week = pd.date_range('2017-11-24', '2017-11-27')                           # 금~월(사이버먼데이)
base_week = [d - pd.Timedelta(weeks=k) for k in range(1, 5) for d in bf_week]
cat_bf = sales_daily[sales_daily.purchase_date.isin(bf_week)].groupby('category_ko').units.sum()
cat_base = sales_daily[sales_daily.purchase_date.isin(base_week)].groupby('category_ko').units.sum() / 4
lift = (cat_bf / cat_base).where(cat_base >= 10).dropna().sort_values(ascending=False)
bf_orders = orders_mart[orders_mart.is_delivered & (orders_mart.purchase_date == bf)]
summ['black_friday'] = {
    'bf_day_orders': int(tot.loc[bf, 'orders']), 'baseline_friday_orders': round(float(tot.loc[base_days, 'orders'].mean()), 1),
    'bf_day_multiple': round(float(tot.loc[bf, 'orders'] / tot.loc[base_days, 'orders'].mean()), 2),
    'bf_week_revenue': round(float(tot.loc[bf_week, 'revenue'].sum()), 2),
    'baseline_week_revenue': round(float(tot.loc[base_week, 'revenue'].sum() / 4), 2),
    'bf_day_late_rate': round(float(bf_orders.late.mean()), 3), 'overall_late_rate': round(float(D.late.mean()), 3),
    'bf_day_mean_score': round(float(bf_orders.review_score.mean()), 2), 'overall_mean_score': round(float(D.review_score.mean()), 2),
    'category_lift_top10': lift.head(10).round(2).to_dict(), 'category_lift_bottom5': lift.tail(5).round(2).to_dict(),
}
# (4·6) 이벤트별 주문 효과: 이벤트 기간 하루 평균 주문 vs 직전 28일(다른 이벤트 날짜 제외) 하루 평균
ev_rows = []
for _, e in events.iterrows():
    win = tot.loc[e.window_start:e.window_end, 'orders'].mean()
    base_idx = [d for d in pd.date_range(e.window_start - pd.Timedelta(days=28), e.window_start - pd.Timedelta(days=1))
                if d not in day_event and d in tot.index]          # 비교 기간에서 다른 이벤트 날짜는 뺌
    base = tot.loc[base_idx, 'orders'].mean()
    if pd.notna(win) and pd.notna(base) and base > 0:
        ev_rows.append({'event_ko': e.event_ko, 'date': str(e.date.date()), 'orders_per_day': round(float(win), 1),
                        'baseline_per_day': round(float(base), 1), 'multiple': round(float(win / base), 2)})
summ['event_effects'] = ev_rows
summ['top_days'] = {str(k.date()): int(v) for k, v in tot.orders.nlargest(6).items()}
print(json.dumps(summ['black_friday'], ensure_ascii=False, indent=1)[:900])
pd.DataFrame(ev_rows)

{
 "bf_day_orders": 1173,
 "baseline_friday_orders": 157.0,
 "bf_day_multiple": 7.47,
 "bf_week_revenue": 306477.16,
 "baseline_week_revenue": 91721.77,
 "bf_day_late_rate": 0.173,
 "overall_late_rate": 0.069,
 "bf_day_mean_score": 3.78,
 "overall_mean_score": 4.15,
 "category_lift_top10": {
  "정원용품": 6.43,
  "침구·욕실·식탁": 5.86,
  "전자제품": 5.33,
  "가방·액세서리": 5.29,
  "향수": 5.18,
  "가구·인테리어": 4.9,
  "장난감": 4.88,
  "게임기·게임": 4.85,
  "휴대폰·통신": 4.67,
  "자동차용품": 4.41
 },
 "category_lift_bottom5": {
  "유아용품": 2.65,
  "시계·선물": 2.44,
  "쿨 아이템": 2.06,
  "문구": 1.92,
  "사무용 가구": 0.52
 }
}


,event_ko,date,orders_per_day,baseline_per_day,multiple
0,블랙프라이데이,2017-11-24,617.8,167.6,3.68
1,크리스마스,2017-12-25,170.1,232.9,0.73
2,어머니날,2017-05-14,111.9,86.4,1.29
3,어머니날,2018-05-13,275.5,239.0,1.15
4,연인의 날(6/12),2017-06-12,120.6,121.0,1.00
5,연인의 날(6/12),2018-06-12,221.1,187.4,1.18
6,아버지날,2017-08-13,130.1,135.8,0.96
7,아버지날,2018-08-12,283.0,222.2,1.27


## 11. 검증
합친 뒤 행 수, 키 중복, 매출 합계가 표마다 일치하는지 확인해요. 여기서 에러가 나면 전처리 어딘가에서 행이 늘거나 빠진 거예요.

In [20]:
checks = {}
checks['orders_mart 주문 수 = 원본'] = len(orders_mart) == len(orders) and orders_mart.order_id.is_unique
checks['order_seller_mart 키 중복 없음'] = not order_seller_mart.duplicated(['order_id', 'seller_id']).any()
checks['item_mart 행 수 = 원본 items'] = len(item_mart) == len(items)
rev_items = sales.price.sum()
rev_orders = orders_mart.loc[orders_mart.is_sale & orders_mart.in_period, 'order_value'].sum()
rev_daily = sales_daily.revenue.sum()
rev_weekly = sales_weekly_seller_cat.revenue.sum()
rev_items_full_weeks = sales[sales.week_start <= LAST_FULL_WEEK].price.sum()
checks['매출 합계 일치 (item·order·daily)'] = np.allclose([rev_orders, rev_daily], rev_items)
checks['주간 표 매출 = 다 채워진 주의 매출'] = bool(np.isclose(rev_weekly, rev_items_full_weeks))
checks['주간 표에 덜 채워진 주 없음'] = bool((sales_weekly_seller_cat.week_start + pd.Timedelta(days=6) <= pd.Timestamp(PERIOD_END)).all())
checks['sales_daily 날짜×카테고리 빠짐 없음'] = len(sales_daily) == len(all_days) * len(all_cats)
checks['review_text 리뷰 수 = 주문당 하나'] = len(review_text) == reviews.order_id.nunique()
checks['정상 행의 발송 처리 시간 음수 없음'] = (order_seller_mart.loc[order_seller_mart.valid_times, 'handling_days'] >= 0).all()
checks['판매자-고객 거리 4,500km 이하 (브라질 안)'] = bool(order_seller_mart.dist_km.max() <= 4500)
srt = order_seller_mart.sort_values('order_purchase_timestamp')
checks['판매자 과거 주문 수가 시간 순으로 줄지 않음 (누수 점검)'] = bool((srt.groupby('seller_id').seller_past_orders.diff().fillna(0) >= 0).all())
first = srt.groupby('seller_id').head(1)
checks['판매자의 첫 주문은 과거 이력 0'] = bool((first.seller_past_orders == 0).all())
checks['seller_stats는 8/15까지 알 수 있던 정보만 (도착·리뷰)'] = bool((v.order_delivered_customer_date < ASOF_END).all() and
    v.loc[v.review_score.notna()].order_id.map(orders_mart.set_index('order_id').review_answer_timestamp).lt(ASOF_END).all())
checks['event_multiples 전체 배수 계산됨'] = bool(event_multiples[event_multiples.category_en == '__all__'].multiple.notna().all())
for k, ok in checks.items():
    print('✅' if ok else '❌', k)
print(f'\n기간 내 매출(상품가 합): {rev_items:,.0f} 헤알')
assert all(checks.values()), '검증 실패 항목이 있어요'

✅ orders_mart 주문 수 = 원본
✅ order_seller_mart 키 중복 없음
✅ item_mart 행 수 = 원본 items
✅ 매출 합계 일치 (item·order·daily)
✅ 주간 표 매출 = 다 채워진 주의 매출
✅ 주간 표에 덜 채워진 주 없음
✅ sales_daily 날짜×카테고리 빠짐 없음
✅ review_text 리뷰 수 = 주문당 하나
✅ 정상 행의 발송 처리 시간 음수 없음
✅ 판매자-고객 거리 4,500km 이하 (브라질 안)
✅ 판매자 과거 주문 수가 시간 순으로 줄지 않음 (누수 점검)
✅ 판매자의 첫 주문은 과거 이력 0
✅ seller_stats는 8/15까지 알 수 있던 정보만 (도착·리뷰)
✅ event_multiples 전체 배수 계산됨

기간 내 매출(상품가 합): 13,195,899 헤알


## 12. 저장 (parquet + JSON + 열 설명)

In [21]:
DICT = {
    'order_seller_mart_purchase_time_cols': ['promised_days', 'dist_km', 'same_state', 'customer_state', 'seller_state', 'n_items', 'price',
                                             'freight_value', 'weight_g', 'volume_cm3', 'category_en', 'seller_past_orders', 'seller_past_late_rate', 'seller_past_score'],
    'orders_mart': {'_grain': '주문 1건', 'purchase_time_cols': ['order_purchase_timestamp', 'year_month', 'weekday', 'hour', 'customer_state',
                    'customer_region', 'n_items', 'order_value', 'freight_value', 'main_category_en', 'payment_type', 'installments', 'promised_days', 'event'],
                    'after_delivery_cols': ['days', 'delay_days', 'late', 'delay_bin', 'review_score', 'low_review', 'answer_hours', 'review_before_delivery'],
                    'days': '구매→도착 일수', 'promised_days': '구매→약속 도착일 일수', 'delay_days': '도착일-약속일 (양수=늦음)',
                    'late': '약속보다 늦게 도착', 'is_sale': '취소·판매불가 아님 (매출 집계용)', 'is_delivered': '배송 완료',
                    'in_period': f'{PERIOD_START} ~ {PERIOD_END}', 'order_value': '상품가 합 (매출)', 'event': '구매일이 속한 쇼핑 이벤트 기간',
                    'answer_hours': '리뷰 요청→고객 답변 시간',
                    'review_before_delivery': '리뷰가 물건 도착 전에 쓰임 (대부분 약속일까지 못 받은 고객)',
                    'delivered_by_asof': '8/15 하루 끝까지 고객에게 도착 (챗봇 현재 상태 계산용)',
                    'review_visible_asof': '8/15 하루 끝까지 리뷰 답변이 달림 (챗봇 현재 상태 계산용)'},
    'order_seller_mart': {'_grain': '주문 × 판매자', 'handling_days': '승인→택배사 인계 (판매자 몫)', 'transit_days': '인계→고객 도착 (택배 몫)',
                          'ship_late': '발송 마감일을 넘겨 인계', 'valid_times': '배송 완료 + 시간 값 정상', 'dist_km': '판매자-고객 거리 (브라질 밖 좌표 제외 후 계산)',
                          'seller_past_orders': '이 주문 구매 시각 이전에 배송이 끝난 이 판매자의 주문 수', 'seller_past_late_rate': '그중 늦게 도착한 비율',
                          'seller_past_score': '구매 시각 이전에 달린 이 판매자 리뷰의 평균', 'free_shipping': '운송비 0원', 'weight_g': '상품 무게 합', 'volume_cm3': '상품 부피 합'},
    'item_mart': {'_grain': '상품 한 줄', 'price': '상품가', 'freight_value': '운송비'},
    'sales_daily': {'_grain': '날짜 × 카테고리 (판매 없는 날 0)', 'units': '판매 수량', 'revenue': '매출'},
    'sales_weekly_seller_cat': {'_grain': '주(월요일 시작) × 판매자 × 카테고리 (첫 판매 주부터 0 채움, 다 채워진 주만)', 'units': '발주 수요 예측의 대상'},
    'product_stats': {'_grain': '상품', 'monthly_units': '월평균 판매량', 'bad_review_cost': '1~2점 리뷰 하나의 예상 월 매출 손해',
                      'reliable': '주문 20건 이상'},
    'seller_stats': {'_grain': '판매자 (8/15까지 도착한 주문·달린 리뷰만으로 계산)', 'peer_*': '같은 대표 카테고리, 주문 30건 이상 판매자의 중앙값', 'score_rank_in_peer': '동종 내 별점 순위',
                     'reliable': f'주문 {MIN_ORDERS_RELIABLE}건 이상'},
    'review_text': {'_grain': '리뷰 (주문당 하나)', 'msg_clean': '소문자·가게 이름 익명화 처리 문장', 'msg_noaccent': '악센트 제거본 (키워드 검색용)',
                    'c_*': '불만 유형 키워드 표시 (글 없는 리뷰는 결측)', 'said_days': '리뷰에 적힌 "N dias" 숫자'},
    'events': {'_grain': '이벤트', 'window_start': '영향 기간 시작', 'window_end': '영향 기간 끝'},
    'event_multiples': {'_grain': '이벤트 × 연도 × 카테고리 (__all__ = 전체)', 'multiple': '이벤트 기간 하루 판매 ÷ 직전 28일(다른 이벤트 제외) 하루 평균. 평소 판매가 적으면 비움',
                        'how_to_use': '미래 이벤트 준비에는 지난해 같은 이벤트의 배수를 씀'},
    'exclusion_log': {'_grain': '처리 단계별 빠지거나 표시된 행 수와 이유'},
    '_config': {'PERIOD': [PERIOD_START, PERIOD_END], 'ASOF': PERIOD_END, 'LOW_SCORE': LOW_SCORE, 'REVIEW_SALES_DROP': REVIEW_SALES_DROP,
                'rule': 'summaries = 8/15까지 구매한 주문의 최종 결과 (연구 분석) / seller_stats·product_stats 별점 = 8/15까지 알 수 있던 정보 (챗봇 현재 상태)'},
}
tables = {'orders_mart': orders_mart, 'order_seller_mart': order_seller_mart, 'item_mart': item_mart, 'sales_daily': sales_daily,
          'sales_weekly_seller_cat': sales_weekly_seller_cat, 'product_stats': product_stats, 'seller_stats': seller_stats,
          'review_text': review_text, 'events': events, 'event_multiples': event_multiples}
for name, t in tables.items():
    t = t.copy()
    for col in t.columns:
        if isinstance(t[col].dtype, pd.CategoricalDtype):
            t[col] = t[col].astype(str).replace('nan', None)
    t.to_parquet(f'{OUT_DIR}/{name}.parquet', index=False)
with open(f'{OUT_DIR}/summaries.json', 'w', encoding='utf-8') as f:
    json.dump(summ, f, ensure_ascii=False, indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
with open(f'{OUT_DIR}/data_dictionary.json', 'w', encoding='utf-8') as f:
    json.dump(DICT, f, ensure_ascii=False, indent=1)
with open(f'{OUT_DIR}/exclusion_log.json', 'w', encoding='utf-8') as f:
    json.dump(EXCLUSIONS, f, ensure_ascii=False, indent=1)

sizes = {f: round(os.path.getsize(f'{OUT_DIR}/{f}') / 1e6, 2) for f in sorted(os.listdir(OUT_DIR))}
print(pd.Series(sizes, name='MB').to_string())
pd.DataFrame(EXCLUSIONS)

data_dictionary.json                0.00
event_multiples.parquet             0.01
events.parquet                      0.00
exclusion_log.json                  0.00
item_mart.parquet                   8.56
order_seller_mart.parquet          17.89
orders_mart.parquet                22.69
product_stats.parquet               1.85
review_text.parquet                14.89
sales_daily.parquet                 0.31
sales_weekly_seller_cat.parquet     0.57
seller_stats.parquet                0.29
summaries.json                      0.01


,step,count,action,reason
0,products,610,미분류로 표시,카테고리 정보가 없는 상품
1,reviews,551,삭제,한 주문에 리뷰가 여러 개 → 마지막에 답한 리뷰만 남김
2,geolocation,42,삭제,브라질 위경도 범위 밖 좌표 (바다·다른 나라로 찍힌 오류)
3,order_items,383,free_shipping 열로 표시,운송비 0원 (무료 배송). 운송비 기준 쿠폰은 0원이 됨
4,orders,306,long_delivery_flag 열로 표시,배송 60일 초과 (지우지 않음)
5,orders,2529,in_period=False로 표시,2017-01-01 ~ 2018-08-15 밖 (시기별 분석에서 제외)
6,orders,2180,in_period=False로 표시,2018-08-16 ~ 08-31: 데이터 수집이 끊겨 주문이 다 들어오지 않은 구간
7,orders,2971,is_delivered=False로 표시,"배송 완료가 아님 (취소, 배송 중 등). 배송·리뷰 분석에서 제외"
8,orders,4976,review_before_delivery 열로 표시,물건 도착 전에 쓰인 리뷰
9,orders,2314,delivered_by_asof=False로 표시,"기간 안에 샀지만 8/16 이후 도착 (연구 분석에는 포함, 챗봇 현재 상태에서는 제외)"


In [22]:
# zip으로 묶기 (팀원 공유용). Colab이면 다운로드 창도 띄움
zip_out = 'olist_mart_v2.zip'   # 압축 안 폴더 이름은 그대로 olist_mart (다른 노트북과 호환)
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT_DIR):
        z.write(f'{OUT_DIR}/{f}', f'olist_mart/{f}')
print(zip_out, round(os.path.getsize(zip_out) / 1e6, 1), 'MB')
if IN_COLAB:
    from google.colab import files
    files.download(zip_out)

olist_mart_v2.zip 55.7 MB


## 13. 다른 노트북·챗봇에서 불러오는 법

```python
import pandas as pd, json
MART = '/content/drive/MyDrive/olist_mart'          # 또는 압축을 푼 olist_mart 폴더
orders_mart = pd.read_parquet(f'{MART}/orders_mart.parquet')
summaries = json.load(open(f'{MART}/summaries.json', encoding='utf-8'))
```

**2판에서 추가된 것 쓰는 법**
- 고객 이해(3) 메뉴: `summaries['review_before_delivery']` → "약속일까지 못 받으면 리뷰 요청이 먼저 가고, 그 리뷰 대부분이 1~2점"
- 예측 모델: `order_seller_mart`의 `seller_past_*`, `weight_g`, `volume_cm3`를 구매 시점 입력으로 쓸 수 있어요 (`data_dictionary.json`의 `order_seller_mart_purchase_time_cols`)
- 발주·이벤트: `event_multiples`에서 지난해 같은 이벤트의 배수를 꺼내 써요
- 발표: `exclusion_log.json`으로 데이터 품질 처리 내역을 표로 보여줄 수 있어요

**기능별로 쓸 표**
1. 배송지연 + 쿠폰: `order_seller_mart`(발송 what-if, 지연 책임), `summaries['delay_vs_score']`(1~2점 확률), `product_stats.bad_review_cost`
2. 평점 + 텍스트: `review_text`(불만 유형, 판매자별·카테고리별로 groupby)
3. 분석 질문: `summaries`의 `same_days_kept_vs_broken`, `state_complaints`, `answer_speed_vs_score`, `said_days`
4. 발주: `sales_weekly_seller_cat`(주별 수요, 0 채움), `events`. 입고까지 걸리는 기간은 판매자가 입력
5. 매출: `sales_daily`, `summaries`의 `monthly_sales`, `category_rank`, `category_pie_top10`
6. 블랙프라이데이: `summaries['black_friday']`, `summaries['event_effects']`, `sales_daily`

**주의**
- 예측 모델에는 `data_dictionary.json`의 `purchase_time_cols`만 입력으로 써요. `after_delivery_cols`는 미래 정보예요.
- 판매자·상품 통계는 `reliable == True`인 것만 비교에 써요.